# Day 5 - Scikit-learn Pipelines

### How can I make my machine learning workflow safer, cleaner, and easier to manage?

When I build a machine learning model, I usually have more than one step to complete. I may need to scale numeric features, encode categorical features, create new features, and then train a model.

But what happens if I do these steps separately or accidentally use information from the test data during training?

This is where Scikit-learn Pipelines become useful. A Pipeline allows me to connect preprocessing steps and the model together in one workflow. This helps me keep the process organized and reduces the risk of data leakage.

During this day, I will start by understanding why Pipelines are important and how they help with data leakage. Then I will learn how to use `ColumnTransformer` to handle numeric and categorical features differently.

After that, I will add engineered features to the workflow and use `GridSearchCV` with 5-fold cross-validation to tune the complete Pipeline.

Finally, I will evaluate the tuned Pipeline on the held-out test set and compare its performance with a baseline model.

By the end of the day, I will have a complete end-to-end machine learning workflow that combines preprocessing, feature engineering, model tuning, and final evaluation.

In [1]:
# for data manipulation and analysis
import pandas as pd

# Create a simple dataset
data = {
    "age": [20, 25, 30, 35, 40, 45, 22, 28, 33, 38],
    "salary": [3000, 4000, 5000, 6000, 7000, 8000, 3500, 4500, 5500, 6500],
    "city": [
        "Nablus",
        "Ramallah",
        "Nablus",
        "Hebron",
        "Ramallah",
        "Nablus",
        "Hebron",
        "Nablus",
        "Ramallah",
        "Hebron"
    ],
    "approved": [0, 1, 1, 1, 1, 1, 0, 1, 1, 1]
}

df = pd.DataFrame(data)

df

,age,salary,city,approved
0,20,3000,Nablus,0
1,25,4000,Ramallah,1
2,30,5000,Nablus,1
3,35,6000,Hebron,1
4,40,7000,Ramallah,1
5,45,8000,Nablus,1
6,22,3500,Hebron,0
7,28,4500,Nablus,1
8,33,5500,Ramallah,1
9,38,6500,Hebron,1


In [2]:
#Define x and y

# Features
X = df.drop("approved", axis=1)

# Target
y = df["approved"]

print("X:")
print(X)

print("\ny:")
print(y)

X:
   age  salary      city
0   20    3000    Nablus
1   25    4000  Ramallah
2   30    5000    Nablus
3   35    6000    Hebron
4   40    7000  Ramallah
5   45    8000    Nablus
6   22    3500    Hebron
7   28    4500    Nablus
8   33    5500  Ramallah
9   38    6500    Hebron

y:
0    0
1    1
2    1
3    1
4    1
5    1
6    0
7    1
8    1
9    1
Name: approved, dtype: int64


In [3]:
# Train/Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Test data:", X_test.shape)

Training data: (8, 3)
Test data: (2, 3)


# Why Pipelines Exist

After learning how to evaluate and tune machine learning models, I learned that I also need to be careful about data leakage.

Data leakage happens when information from validation or test data accidentally becomes available during the training process.

For example, if I scale the whole dataset before splitting it, the scaler can learn information from data that should have remained unseen.

To avoid this problem, I can use a Scikit-learn Pipeline.

A Pipeline connects preprocessing and the machine learning model together in one object.

This means that when I train the Pipeline, the preprocessing is fitted only on the training data. When I make predictions, the same preprocessing is applied to the new data.

This makes the workflow safer and helps me avoid data leakage.


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Create a simple Pipeline
pipe_basic = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])

# Building a Pipeline

After understanding why I need a Pipeline, I moved to building one.

A Pipeline contains multiple steps, and each step has a name and an object that performs a specific task.

In this example, I use a preprocessing step followed by a Random Forest model.

The main idea is that I do not have to manually run every step separately.

When I call `fit()`, the Pipeline runs the steps in the correct order.

When I call `predict()`, it applies the same preprocessing before making the prediction.

This makes my machine learning workflow cleaner and easier to manage.


In [5]:
pipe_basic.fit(X_train[["age", "salary"]], y_train)

predictions = pipe_basic.predict(X_test[["age", "salary"]])

print("Predictions:")
print(predictions)

Predictions:
[1 0]


# ColumnTransformer for Mixed Data

After building a basic Pipeline, I noticed that my dataset contains different types of features.

The `age` and `salary` columns are numeric, while the `city` column is categorical.

These columns need different preprocessing.

For the numeric columns, I use `StandardScaler`.

For the categorical column, I use `OneHotEncoder`.

To apply these preprocessing steps correctly, I use `ColumnTransformer`.

ColumnTransformer allows me to define a different preprocessing method for each group of columns.

Then I can place the ColumnTransformer inside my Pipeline before the machine learning model.

This gives me one complete workflow that can handle both numeric and categorical data.


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define column types
numeric_cols = ["age", "salary"]
categorical_cols = ["city"]

# Create ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

# Create the final Pipeline
pipe = Pipeline([
    ("pre", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

# Train
pipe.fit(X_train, y_train)

# Predict
predictions = pipe.predict(X_test)

print("Predictions:")
print(predictions)

Predictions:
[1 0]


# Tuning a Whole Pipeline

After creating the complete Pipeline, I can now tune the model.

Instead of manually choosing the hyperparameters, I use GridSearchCV.

GridSearchCV tries different combinations of hyperparameters and evaluates them using cross-validation.

Because my model is inside a Pipeline, I can tune its parameters using the name of the step followed by two underscores.

For example, `model__n_estimators` refers to the `n_estimators` parameter inside my model step.

I use 5-fold cross-validation so that the model is evaluated on different parts of the training data.

After the search finishes, I can get the best parameters and the best cross-validation score.


In [7]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [3, 5, 10]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="f1"
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(grid.best_score_)

d:\Anaconda\envs\ai-ml\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Best Parameters:
{'model__max_depth': 3, 'model__n_estimators': 100}

Best Cross-Validation F1 Score:
0.9333333333333332


In [8]:
best_pipeline = grid.best_estimator_

print("Best Pipeline:")
print(best_pipeline)

Best Pipeline:
Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'salary']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city'])])),
                ('model',
                 RandomForestClassifier(max_depth=3, random_state=42))])


# Week 4 Mini-Project

Now I can combine everything I learned during Day 5 into one complete workflow.

I start with my data and create an engineered feature.

Then I use ColumnTransformer to scale the numeric features and encode the categorical features.

After that, I put the preprocessing and the Random Forest model inside one Pipeline.

I use GridSearchCV with 5-fold cross-validation to find the best hyperparameters.

Finally, after all the tuning is finished, I evaluate the best Pipeline on the held-out test set.

This gives me a complete machine learning workflow that is organized, reproducible, and designed to avoid data leakage.


# Hands-On Lab: Tuned End-to-End Pipeline

# Task 1 - Build a Pipeline with ColumnTransformer

In this task, I built a Pipeline that handles both numeric and categorical features.

I used StandardScaler for the numeric columns and OneHotEncoder for the categorical column.

Then I placed the ColumnTransformer and the Random Forest model inside one Pipeline.

This allows the preprocessing and the model to work together as one complete workflow.


In [9]:
# Numeric columns
numeric_cols = ["age", "salary"]

# Categorical columns
categorical_cols = ["city"]

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

# Pipeline
pipe = Pipeline([
    ("pre", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

# Train
pipe.fit(X_train, y_train)

print("Pipeline completed successfully.")

Pipeline completed successfully.


# Task 2 - Add Engineered Features

In this task, I added an engineered feature to my dataset.

I created `salary_per_age` by dividing salary by age.

The purpose of this feature is to give the model additional information derived from the existing features.

I added the new feature to both the training and test data.

Then I included it in the numeric features handled by the ColumnTransformer.

Now the Pipeline can use the original features together with the engineered feature.


In [10]:
# Copy the datasets
X_train = X_train.copy()
X_test = X_test.copy()

# Create the engineered feature
X_train["salary_per_age"] = X_train["salary"] / X_train["age"]
X_test["salary_per_age"] = X_test["salary"] / X_test["age"]

# Update feature lists
numeric_cols = [
    "age",
    "salary",
    "salary_per_age"
]

categorical_cols = ["city"]

# Create preprocessing
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

# Create Pipeline
pipe = Pipeline([
    ("pre", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

# Train
pipe.fit(X_train, y_train)

print("Engineered feature added successfully.")

Engineered feature added successfully.


# Task 3 - Tune the Full Pipeline

In this task, I tuned the complete Pipeline using GridSearchCV.

I defined different values for the Random Forest hyperparameters.

GridSearchCV tested these combinations using 5-fold cross-validation.

Because I passed the complete Pipeline to GridSearchCV, the preprocessing and the model were evaluated together.

After the search finished, I selected the best Pipeline based on the cross-validation score.


In [15]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [3, 5, 10]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="f1"
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(grid.best_score_)

d:\Anaconda\envs\ai-ml\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Best Parameters:
{'model__max_depth': 3, 'model__n_estimators': 50}

Best Cross-Validation F1 Score:
0.9333333333333332


In [16]:
best_pipeline = grid.best_estimator_

print("Best Pipeline:")
print(best_pipeline)

Best Pipeline:
Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'salary',
                                                   'salary_per_age']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city'])])),
                ('model',
                 RandomForestClassifier(max_depth=3, n_estimators=50,
                                        random_state=42))])


# Task 4 - Evaluate the Final Pipeline

In this task, I compared the tuned Pipeline with a baseline model.

First, I trained the baseline Pipeline without hyperparameter tuning.

Then I used the tuned Pipeline that I obtained from GridSearchCV.

I evaluated both models on the held-out test set using the F1 score.

The test set was kept until the end because I wanted to use it only for the final evaluation.

Finally, I compared the two scores to see whether tuning improved the model.


In [12]:
from sklearn.metrics import f1_score

# Create baseline Pipeline
baseline = Pipeline([
    ("pre", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

# Train baseline
baseline.fit(X_train, y_train)

# Baseline predictions
baseline_pred = baseline.predict(X_test)

# Baseline score
baseline_score = f1_score(y_test, baseline_pred)

print("Baseline F1 Score:")
print(baseline_score)

Baseline F1 Score:
1.0


In [13]:
#Tuned Model

# Predictions from the tuned Pipeline
tuned_pred = best_pipeline.predict(X_test)

# Tuned score
tuned_score = f1_score(y_test, tuned_pred)

print("Tuned Pipeline F1 Score:")
print(tuned_score)

Tuned Pipeline F1 Score:
0.6666666666666666


In [14]:
#Comparison

print("Baseline F1 Score:", baseline_score)
print("Tuned F1 Score:", tuned_score)
print("Improvement:", tuned_score - baseline_score)

Baseline F1 Score: 1.0
Tuned F1 Score: 0.6666666666666666
Improvement: -0.33333333333333337
